# M5 Forecasting - Complete Pipeline (Colab)
## CS 415 Deep Learning Final Project

**Team:** Suat Emre Karabıçak, Alpay Naçar

---

### Features:
- All 7 models (RNN, GRU, LSTM, TCN, Informer, Autoformer, FEDformer)
- Mixed Precision Training (FP16) for 2x speedup on A100
- Smart Resume (skips completed models)
- Auto-backup to Google Drive
- RMSE and WRMSSE metrics

### Expected Runtime on A100:
- With FP16: ~3.5 hours
- Without FP16: ~6 hours

## Step 1: Setup Environment

In [ ]:
# Check GPU
!nvidia-smi

Mon Dec 29 14:13:08 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   42C    P0             76W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
# Install dependencies (if needed)
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q numpy pandas scikit-learn matplotlib tqdm

In [3]:
# Mount Google Drive for data and backups
from google.colab import drive
drive.mount('/content/drive')

# Create directories
!mkdir -p data
!mkdir -p /content/drive/MyDrive/m5_results

print("✓ Google Drive mounted")
print("✓ Directories created")

Mounted at /content/drive
✓ Google Drive mounted
✓ Directories created


## Step 2: Upload Python Files and Data

**Upload these 5 Python files to Colab:**
1. `data_preparation_final.py`
2. `models_final.py`
3. `wrmsse_metric.py`
4. `training_final.py`
5. `main_pipeline_final.py`

**Upload M5 dataset to `data/` folder:**
- `sales_train_evaluation.csv`
- `calendar.csv`
- `sell_prices.csv`

**Or copy from Google Drive if already uploaded:**

In [5]:
# Option: Copy data from Google Drive (if you uploaded there)
# Uncomment and modify paths as needed:

# !cp /content/drive/MyDrive/m5_data/*.csv data/
# print("✓ Data copied from Drive")

# Verify data files exist
import os
required_files = [
    'data/sales_train_evaluation.csv',
    'data/calendar.csv',
    'data/sell_prices.csv'
]

for file in required_files:
    if os.path.exists(file):
        print(f"✓ Found: {file}")
    else:
        print(f"❌ Missing: {file} - Please upload!")

✓ Found: data/sales_train_evaluation.csv
✓ Found: data/calendar.csv
✓ Found: data/sell_prices.csv


In [6]:
# Verify Python modules are uploaded
python_modules = [
    'data_preparation_final.py',
    'models_final.py',
    'wrmsse_metric.py',
    'training_final.py',
    'main_pipeline_final.py'
]

all_present = True
for module in python_modules:
    if os.path.exists(module):
        print(f"✓ Found: {module}")
    else:
        print(f"❌ Missing: {module} - Please upload!")
        all_present = False

if all_present:
    print("\n✓ All files ready! You can proceed.")
else:
    print("\n⚠️ Please upload missing files before continuing.")

✓ Found: data_preparation_final.py
✓ Found: models_final.py
✓ Found: wrmsse_metric.py
✓ Found: training_final.py
❌ Missing: main_pipeline_final.py - Please upload!

⚠️ Please upload missing files before continuing.


## Step 3: Import Modules and Setup

In [ ]:
# Import standard libraries
import torch

# Enable TensorCore acceleration for FP32 ops on A100\n",
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.set_float32_matmul_precision("high")

import numpy as np
import random
import os
import sys
import pickle
import json
from torch.utils.data import DataLoader
from sklearn.metrics import mean_squared_error

# Import our custom modules
from data_preparation_final import M5DataPreprocessor
from models_final import SimpleRNN, GRU, LSTM, TCN, Informer, Autoformer, FEDformer
from training_final import (
    TimeSeriesDataset, Trainer,
    compare_models, save_results_json, create_results_table,
    diagnose_data
)
from wrmsse_metric import WRMSSECalculator, simple_rmse

print("✓ All modules imported successfully")
print(f"✓ PyTorch version: {torch.__version__}")
print(f"✓ CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✓ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✓ GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

✓ All modules imported successfully
✓ PyTorch version: 2.9.0+cu126
✓ CUDA available: True
✓ GPU: NVIDIA A100-SXM4-80GB
✓ GPU Memory: 85.2 GB


## Step 4: Configuration

**Modify these settings as needed:**

In [3]:
# ============================================================================
# CONFIGURATION - OPTIMIZED
# ============================================================================

CONFIG = {
    'batch_size': 16384,
    'epochs': 30,
    'early_stopping_patience': 5,
    'output_length': 28,
    'precision': 'bf16',
    'seed': 42,
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    'n_series': None,
    'input_length': 90,
    'stride': 7,
    'train_ratio': 0.7,
    'val_ratio': 0.15,
    'test_ratio': 0.15
}

MODEL_LR = {
    'RNN': 0.002,
    'GRU': 0.002,
    'LSTM': 0.002,
    'TCN': 0.001,
    'Informer': 0.0005,
    'Autoformer': 0.0005,
    'FEDformer': 0.0005
}

MODEL_CONFIGS = {
    'RNN': {'hidden_size': 256, 'num_layers': 2, 'dropout': 0.1},
    'GRU': {'hidden_size': 256, 'num_layers': 2, 'dropout': 0.1},
    'LSTM': {'hidden_size': 256, 'num_layers': 2, 'dropout': 0.1},
    'TCN': {'num_channels': [128, 256, 256], 'kernel_size': 3, 'dropout': 0.1},
    'Informer': {'d_model': 256, 'n_heads': 8, 'e_layers': 2, 'd_ff': 1024, 'dropout': 0.1, 'factor': 5},
    'Autoformer': {'d_model': 256, 'n_heads': 8, 'e_layers': 2, 'd_ff': 1024, 'dropout': 0.1},
    'FEDformer': {'d_model': 256, 'modes': 32, 'e_layers': 2, 'd_ff': 1024, 'dropout': 0.1}
}

MODEL_CLASSES = {
    'RNN': SimpleRNN,
    'GRU': GRU,
    'LSTM': LSTM,
    'TCN': TCN,
    'Informer': Informer,
    'Autoformer': Autoformer,
    'FEDformer': FEDformer
}

# SELECT MODELS TO TRAIN
MODELS_TO_TRAIN = ['GRU', 'TCN', 'Autoformer']  # Fast: ~1.5 hours
# MODELS_TO_TRAIN = ['RNN', 'GRU', 'LSTM', 'TCN', 'Informer', 'Autoformer', 'FEDformer']  # All: ~4 hours

print(f"Models to train: {MODELS_TO_TRAIN}")
print(f"Config: {CONFIG}")

Models to train: ['GRU', 'TCN', 'Autoformer']
Config: {'batch_size': 16384, 'epochs': 30, 'early_stopping_patience': 5, 'output_length': 28, 'precision': 'bf16', 'seed': 42, 'device': 'cuda'}


## Step 5: Set Random Seeds

In [4]:
def set_seed(seed=42):
    """Set random seeds for reproducibility."""
    print(f"\n{'='*80}")
    print(f"SETTING RANDOM SEEDS FOR REPRODUCIBILITY")
    print(f"{'='*80}")
    print(f"Seed: {seed}")

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.benchmark = True  # Keep for speed

    print("✓ Seeds set (reasonable reproducibility mode)")
    print("  - Python, NumPy, PyTorch: Fixed")
    print("  - CUDA: Seeded with benchmark enabled")
    print("  - Expected variation: ±0.01 RMSE between runs")

set_seed(CONFIG['seed'])


SETTING RANDOM SEEDS FOR REPRODUCIBILITY
Seed: 42
✓ Seeds set (reasonable reproducibility mode)
  - Python, NumPy, PyTorch: Fixed
  - CUDA: Seeded with benchmark enabled
  - Expected variation: ±0.01 RMSE between runs


In [10]:
# Copy data files from your Drive backup
import shutil
import os

backup_path = '/content/drive/MyDrive/m5_backup_20251229_0033'

# Copy all .npy data files
data_files = [
    'X_train.npy', 'X_val.npy', 'X_test.npy',
    'Y_train.npy', 'Y_val.npy', 'Y_test.npy',
    'feature_cols.txt', 'scaler.pkl', 'df_clean_for_wrmsse.pkl'
]

print("Copying data files from Drive...")
for f in data_files:
    src = os.path.join(backup_path, f)
    if os.path.exists(src):
        shutil.copy(src, '/content/')
        print(f"  ✓ {f}")
    else:
        print(f"  ❌ {f} not found!")

print("\n✓ Data files ready!")
print("\nNOTE: Upload the FIXED Python files I provided:")
print("  - models_final.py (FIXED)")
print("  - training_final.py (FIXED)")
print("  - wrmsse_metric.py (FIXED)")
print("Do NOT use the old ones from your backup!")

Copying data files from Drive...
  ✓ X_train.npy
  ✓ X_val.npy
  ✓ X_test.npy
  ✓ Y_train.npy
  ✓ Y_val.npy
  ✓ Y_test.npy
  ✓ feature_cols.txt
  ✓ scaler.pkl
  ✓ df_clean_for_wrmsse.pkl

✓ Data files ready!

NOTE: Upload the FIXED Python files I provided:
  - models_final.py (FIXED)
  - training_final.py (FIXED)
  - wrmsse_metric.py (FIXED)
Do NOT use the old ones from your backup!


## Step 6: Data Preparation

This step will:
1. Load and merge M5 data
2. Create 23 engineered features
3. Generate sliding window sequences
4. Normalize with RobustScaler
5. Split into train/val/test

**Time:** 30-60 minutes for full dataset

In [5]:
print("\n" + "="*80)
print("STEP 1: DATA PREPARATION")
print("="*80)

# Check if preprocessed data already exists
if os.path.exists('X_train.npy') and os.path.exists('df_clean_for_wrmsse.pkl'):
    print("\n✓ Found preprocessed data, loading...")

    X_train = np.load('X_train.npy')
    Y_train = np.load('Y_train.npy')
    X_val = np.load('X_val.npy')
    Y_val = np.load('Y_val.npy')
    X_test = np.load('X_test.npy')
    Y_test = np.load('Y_test.npy')

    with open('feature_cols.txt', 'r') as f:
        feature_cols = f.read().split('\n')

    df_clean = pickle.load(open('df_clean_for_wrmsse.pkl', 'rb'))

    print(f"  X_train: {X_train.shape}")
    print(f"  X_val: {X_val.shape}")
    print(f"  X_test: {X_test.shape}")
    print(f"  Features: {len(feature_cols)}")

else:
    print("\n⚠ Preprocessed data not found, running full pipeline...")
    print("  This will take 30-60 minutes for full dataset...\n")

    # Initialize preprocessor
    preprocessor = M5DataPreprocessor(
        sales_path="data/sales_train_evaluation.csv",
        calendar_path="data/calendar.csv",
        prices_path="data/sell_prices.csv"
    )

    # Load data
    df = preprocessor.load_data(n_series=CONFIG['n_series'])

    # Create features
    df = preprocessor.create_features(df)

    # Create sequences
    X, Y, feature_cols, df_clean = preprocessor.create_sequences(
        df,
        input_length=CONFIG['input_length'],
        output_length=CONFIG['output_length'],
        stride=CONFIG['stride']
    )

    # Split data
    X_train, Y_train, X_val, Y_val, X_test, Y_test = preprocessor.split_data(
        X, Y,
        train_ratio=CONFIG['train_ratio'],
        val_ratio=CONFIG['val_ratio'],
        test_ratio=CONFIG['test_ratio']
    )

    # Normalize with RobustScaler
    X_train, X_val, X_test = preprocessor.normalize_features(X_train, X_val, X_test)

    # Save everything
    print("\nSaving preprocessed data...")
    np.save('X_train.npy', X_train)
    np.save('Y_train.npy', Y_train)
    np.save('X_val.npy', X_val)
    np.save('Y_val.npy', Y_val)
    np.save('X_test.npy', X_test)
    np.save('Y_test.npy', Y_test)

    with open('feature_cols.txt', 'w') as f:
        f.write('\n'.join(feature_cols))

    preprocessor.save_scaler('scaler.pkl')
    df_clean.to_pickle('df_clean_for_wrmsse.pkl')

    print("✓ Preprocessing complete and saved")

print("NOTE: Y_*.npy are log1p-scaled targets (use np.expm1 to recover original units).")
print("\n✓ Data preparation complete!")


STEP 1: DATA PREPARATION

✓ Found preprocessed data, loading...
  X_train: (5485151, 90, 23)
  X_val: (1175389, 90, 23)
  X_test: (1175390, 90, 23)
  Features: 23

✓ Data preparation complete!


In [12]:
# Data diagnostics
diagnose_data(X_train, Y_train, X_val, Y_val)


DATA DIAGNOSTICS

📊 Training Data Statistics:
  X_train shape: (5485151, 90, 23)
  X_train - mean: -926.3668, std: 17743.1758
  X_train - min: -120698.9688, max: 3258852.5000
  X_train has NaN: False
  X_train has Inf: False

  Y_train shape: (5485151, 28)
  Y_train - mean: 1.1888, std: 4.0352
  Y_train - min: 0.0000, max: 763.0000
  Y_train has NaN: False
  Y_train has Inf: False

  Y_train zero percentage: 66.6%

🚨 POTENTIAL ISSUES DETECTED:
  ⚠️ X_train has very high variance - consider additional normalization
  ⚠️ 66.6% of targets are zero - sparse data, consider special handling

📊 Validation Data Statistics:
  X_val shape: (1175389, 90, 23)
  Y_val shape: (1175389, 28)


False

In [14]:
# ============================================================================
# FIX: Clip extreme values and re-normalize X
# ============================================================================
print("Applying additional normalization...")

# Clip extreme values (per feature, to 1st-99th percentile)
for i in range(X_train.shape[2]):  # For each feature
    p1 = np.percentile(X_train[:, :, i], 1)
    p99 = np.percentile(X_train[:, :, i], 99)

    X_train[:, :, i] = np.clip(X_train[:, :, i], p1, p99)
    X_val[:, :, i] = np.clip(X_val[:, :, i], p1, p99)
    X_test[:, :, i] = np.clip(X_test[:, :, i], p1, p99)

# Re-normalize to [-1, 1] range
for i in range(X_train.shape[2]):
    min_val = X_train[:, :, i].min()
    max_val = X_train[:, :, i].max()

    if max_val - min_val > 0:
        X_train[:, :, i] = 2 * (X_train[:, :, i] - min_val) / (max_val - min_val) - 1
        X_val[:, :, i] = 2 * (X_val[:, :, i] - min_val) / (max_val - min_val) - 1
        X_test[:, :, i] = 2 * (X_test[:, :, i] - min_val) / (max_val - min_val) - 1

# Verify
print(f"After fix:")
print(f"  X_train - mean: {X_train.mean():.4f}, std: {X_train.std():.4f}")
print(f"  X_train - min: {X_train.min():.4f}, max: {X_train.max():.4f}")

Applying additional normalization...
After fix:
  X_train - mean: -0.4854, std: 0.6359
  X_train - min: -1.0000, max: 1.0000


## Step 7: Create DataLoaders

In [ ]:
print("\n" + "="*80)
print("STEP 2: CREATING DATALOADERS")
print("="*80)

# Create DataLoaders
train_dataset = TimeSeriesDataset(X_train, Y_train)
val_dataset = TimeSeriesDataset(X_val, Y_val)
test_dataset = TimeSeriesDataset(X_test, Y_test)

num_workers = min(8, os.cpu_count() or 1)
use_persistent = (os.cpu_count() or 1) > 1

train_loader = DataLoader(
    train_dataset,
    batch_size=CONFIG['batch_size'],
    shuffle=True,
    num_workers=num_workers,
    pin_memory=True,
    prefetch_factor=4,
    persistent_workers=use_persistent
)

val_loader = DataLoader(
    val_dataset,
    batch_size=CONFIG['batch_size'],
    shuffle=False,
    num_workers=num_workers,
    pin_memory=True,
    prefetch_factor=4,
    persistent_workers=use_persistent
)

test_loader = DataLoader(
    test_dataset,
    batch_size=CONFIG['batch_size'],
    shuffle=False,
    num_workers=num_workers,
    pin_memory=True,
    prefetch_factor=4,
    persistent_workers=use_persistent
)

print(f"✓ Train batches: {len(train_loader)}")
print(f"✓ Val batches: {len(val_loader)}")
print(f"✓ Test batches: {len(test_loader)}")


STEP 2: CREATING DATALOADERS
✓ Train batches: 335
✓ Val batches: 72
✓ Test batches: 72


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 16 worker processes in total. Our suggested max number of worker in current system is 12, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


## Step 8: Initialize WRMSSE Calculator

In [ ]:
print("\n" + "="*80)
print("STEP 3: INITIALIZING WRMSSE CALCULATOR")
print("="*80)

wrmsse_calc = WRMSSECalculator(df_clean)

print("\n✓ WRMSSE calculator ready")


STEP 3: INITIALIZING WRMSSE CALCULATOR

INITIALIZING WRMSSE CALCULATOR

Calculating scale factors...
  ✓ Calculated scale factors for 30490 series
  Scale factor range: [0.0000, 45.6739]

Calculating dollar-based weights...
  ✓ Calculated weights for 30490 series
  Total dollar sales: $189,571,488.00
  Weight range: [0.000000, 0.002375]
✓ WRMSSE calculator ready

✓ WRMSSE calculator ready


## Step 10: Helper Functions

In [7]:
def backup_to_drive(files_to_backup):
    """Backup files to Google Drive."""
    drive_path = '/content/drive/MyDrive/m5_results'

    if not os.path.exists('/content/drive'):
        print("⚠️ Google Drive not mounted, skipping backup")
        return

    os.makedirs(drive_path, exist_ok=True)

    import shutil
    backed_up = 0
    for file in files_to_backup:
        if os.path.exists(file):
            try:
                shutil.copy(file, drive_path)
                backed_up += 1
            except:
                pass

    if backed_up > 0:
        print(f"✓ Backed up {backed_up} files to Drive")


def check_model_completed(model_name):
    """Check if model training already completed."""
    checkpoint_file = f'best_{model_name}.pt'
    results_file = f'{model_name}_predictions.npy'

    return os.path.exists(checkpoint_file) and os.path.exists(results_file)


def load_completed_model_results(model_name):
    """Load results from completed model."""
    predictions = np.load(f'{model_name}_predictions.npy')
    targets = np.load(f'{model_name}_targets.npy')
    rmse = np.sqrt(mean_squared_error(targets, predictions))

    return {
        'rmse': rmse,
        'predictions': predictions,
        'targets': targets
    }

print("✓ Helper functions defined")

✓ Helper functions defined


In [21]:
# ============================================
# SAVE ALL PROGRESS TO GOOGLE DRIVE
# ============================================

import shutil
import os
from datetime import datetime

# Create backup folder with timestamp
timestamp = datetime.now().strftime("%Y%m%d_%H%M")
backup_path = f'/content/drive/MyDrive/m5_backup_{timestamp}'
os.makedirs(backup_path, exist_ok=True)

print(f"Backing up to: {backup_path}")
print("="*60)

# Files to backup
files_to_backup = [
    # Preprocessed data (the expensive stuff!)
    'X_train.npy',
    'Y_train.npy',
    'X_val.npy',
    'Y_val.npy',
    'X_test.npy',
    'Y_test.npy',
    'feature_cols.txt',
    'scaler.pkl',
    'df_clean_for_wrmsse.pkl',

    # Python files
    'data_preparation_final.py',
    'models_final.py',
    'wrmsse_metric.py',
    'training_final.py',
    'main_pipeline_final.py',

    # Config
    'config.json' if os.path.exists('config.json') else None,
]

# Backup each file
backed_up = 0
total_size = 0

for file in files_to_backup:
    if file and os.path.exists(file):
        file_size = os.path.getsize(file) / (1024**3)  # GB
        print(f"  Copying {file} ({file_size:.2f} GB)...")
        shutil.copy(file, backup_path)
        backed_up += 1
        total_size += file_size
    elif file:
        print(f"  ⚠️  Skipping {file} (not found)")

print("="*60)
print(f"✓ Backed up {backed_up} files ({total_size:.2f} GB total)")
print(f"✓ Location: {backup_path}")
print("\n✓ You can safely stop the notebook now!")
print("✓ To resume: Upload these files back to Colab")

Backing up to: /content/drive/MyDrive/m5_backup_20251229_1318
  Copying X_train.npy (42.30 GB)...
  Copying Y_train.npy (0.57 GB)...
  Copying X_val.npy (9.06 GB)...
  Copying Y_val.npy (0.12 GB)...
  Copying X_test.npy (9.06 GB)...
  Copying Y_test.npy (0.12 GB)...
  Copying feature_cols.txt (0.00 GB)...
  Copying scaler.pkl (0.00 GB)...
  Copying df_clean_for_wrmsse.pkl (5.43 GB)...
  Copying data_preparation_final.py (0.00 GB)...
  Copying models_final.py (0.00 GB)...
  Copying wrmsse_metric.py (0.00 GB)...
  Copying training_final.py (0.00 GB)...
  ⚠️  Skipping main_pipeline_final.py (not found)
✓ Backed up 13 files (66.67 GB total)
✓ Location: /content/drive/MyDrive/m5_backup_20251229_1318

✓ You can safely stop the notebook now!
✓ To resume: Upload these files back to Colab


## Step 11: Train All Models

**This is the main training loop!**

Features:
- Smart resume (skips completed models)
- Auto-backup after each model
- Progress tracking

**Time:** ~3.5 hours with FP16 on A100

In [8]:
print("\n" + "="*80)
print("STEP 4: MODEL TRAINING")
print("="*80)

input_dim = X_train.shape[2]
output_len = CONFIG['output_length']

# Store results
results = {}
training_times = {}
histories = {}

# Model classes mapping
MODEL_CLASSES = {
    #'RNN': SimpleRNN,
    'GRU': GRU,
    #'LSTM': LSTM,
    'TCN': TCN,
    #'Informer': Informer,
    'Autoformer': Autoformer,
    #'FEDformer': FEDformer
}

print(f"\nTraining {len(MODEL_CLASSES)} models...\n")


STEP 4: MODEL TRAINING

Training 3 models...



In [ ]:
# Train each model
import time

# Train each model
for model_name in MODELS_TO_TRAIN:

    # Check if already done
    if check_model_completed(model_name):
        print(f"\n{'='*60}")
        print(f"SKIPPING {model_name} - Already trained!")
        print(f"{'='*60}")
        results[model_name] = load_completed_model_results(model_name)
        print(f"  Loaded RMSE: {results[model_name]['rmse']:.4f}")
        continue

    # Train
    print(f"\n{'='*60}")
    print(f"TRAINING: {model_name}")
    print(f"{'='*60}")

    ModelClass = MODEL_CLASSES[model_name]
    lr = MODEL_LR[model_name]
    config = MODEL_CONFIGS[model_name].copy()

    print(f"Config: {config}")
    print(f"LR: {lr}")

    # Create model
    model = ModelClass(input_dim=input_dim, output_len=output_len, **config)
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Parameters: {n_params:,}")

    # Create trainer
    trainer = Trainer(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        test_loader=test_loader,
        lr=lr,
        device=CONFIG['device'],
        model_name=model_name,
        precision=CONFIG['precision']
    )

    # Train
    history, train_time = trainer.train(
        epochs=CONFIG['epochs'],
        early_stopping_patience=CONFIG['early_stopping_patience']
    )

    training_times[model_name] = train_time
    histories[model_name] = history

    # Test
    test_results = trainer.test()
    results[model_name] = test_results

    # Save
    np.save(f'{model_name}_predictions.npy', test_results['predictions'])
    np.save(f'{model_name}_targets.npy', test_results['targets'])
    trainer.plot_history(save_path=f'{model_name}_history.png')

    # Backup
    backup_to_drive([
        f'best_{model_name}.pt',
        f'{model_name}_predictions.npy',
        f'{model_name}_targets.npy',
        f'{model_name}_history.png'
    ])

    print(f"\n✓ {model_name} complete")
    print(f"  Test RMSE: {test_results['rmse']:.4f}")
    print(f"  Time: {train_time/60:.1f} min")

    # Clear memory
    del trainer, model
    torch.cuda.empty_cache()
print("\n" + "="*80)
print("ALL TRAINING COMPLETE!")
print("="*80)


SKIPPING GRU - Already trained!
  Loaded RMSE: 3.0258

SKIPPING TCN - Already trained!
  Loaded RMSE: 2.9613

TRAINING: Autoformer
Config: {'d_model': 256, 'n_heads': 8, 'e_layers': 2, 'd_ff': 1024, 'dropout': 0.1}
LR: 0.0005
Parameters: 1,591,996

TRAINING: Autoformer
  FP16 enabled: False
  Initial LR: 0.0005

Epoch 1/30


Training:   0%|          | 0/335 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 16 worker processes in total. Our suggested max number of worker in current system is 12, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(
Training:  38%|███▊      | 126/335 [04:14<06:49,  1.96s/it, loss=634]

## Step 12: Calculate WRMSSE

In [ ]:
print("\n" + "="*80)
print("STEP 5: CALCULATING WRMSSE")
print("="*80)

# Create series ID mapping for test set
series_ids_test = np.array([
    df_clean['id'].unique()[i % len(df_clean['id'].unique())]
    for i in range(len(Y_test))
])

for model_name in results.keys():
    print(f"\n{model_name}:")
    predictions = np.expm1(results[model_name]['predictions'])
    actuals = np.expm1(results[model_name]['targets'])

    wrmsse = wrmsse_calc.calculate_wrmsse(predictions, actuals, series_ids_test)
    results[model_name]['wrmsse'] = wrmsse

print("\n✓ WRMSSE calculated for all models")

## Step 13: Results Analysis

In [ ]:
print("\n" + "="*80)
print("STEP 6: RESULTS ANALYSIS")
print("="*80)

# Print results table
create_results_table(results)

# Save results
save_results_json(results, training_times, save_path='final_results.json')

# Compare models
compare_models(results, save_path='model_comparison.png')

# Find best model
best_model_name = min(results.items(), key=lambda x: x[1]['rmse'])[0]
print(f"\n🏆 BEST MODEL: {best_model_name}")
print(f"  RMSE: {results[best_model_name]['rmse']:.4f}")
print(f"  WRMSSE: {results[best_model_name]['wrmsse']:.4f}")

# Backup final results
backup_to_drive([
    'final_results.json',
    'model_comparison.png'
])

## Step 14: Save Everything

In [ ]:
print("\n" + "="*80)
print("STEP 7: SAVING ALL OUTPUTS")
print("="*80)

# Save histories
with open('training_histories.pkl', 'wb') as f:
    pickle.dump(histories, f)
print("✓ Saved training histories")

# Save config
config_save = {k: str(v) if not isinstance(v, (int, float, str, bool, type(None))) else v
               for k, v in CONFIG.items()}
with open('config.json', 'w') as f:
    json.dump(config_save, f, indent=2)
print("✓ Saved configuration")

# Final backup
backup_to_drive([
    'training_histories.pkl',
    'config.json',
    'final_results.json'
])

print("\n" + "="*80)
print("PIPELINE COMPLETE!")
print("="*80)

print("\n✓ All results saved to:")
print("  - Local: /content/")
print("  - Google Drive: /content/drive/MyDrive/m5_results/")

print("\n✓ Ready for final report!")

## Step 15: View Results Summary

In [ ]:
# Display final results in nice format
import pandas as pd

results_df = pd.DataFrame([
    {
        'Model': name,
        'RMSE': f"{res['rmse']:.4f}",
        'WRMSSE': f"{res['wrmsse']:.4f}",
        'Training Time (min)': f"{training_times.get(name, 0)/60:.1f}"
    }
    for name, res in sorted(results.items(), key=lambda x: x[1]['rmse'])
])

print("\n" + "="*80)
print("FINAL RESULTS SUMMARY")
print("="*80)
print(results_df.to_string(index=False))

# Calculate improvement over baseline
baseline_rmse = results['GRU']['rmse']
best_rmse = min(res['rmse'] for res in results.values())
improvement = (baseline_rmse - best_rmse) / baseline_rmse * 100

print(f"\n✓ Improvement over GRU baseline: {improvement:.2f}%")
print(f"✓ Best model: {best_model_name}")

## Step 16: Display Visualizations

In [ ]:
# Display model comparison plot
from IPython.display import Image, display

if os.path.exists('model_comparison.png'):
    print("Model Comparison:")
    display(Image('model_comparison.png'))

In [ ]:
# Display best model training history
if os.path.exists(f'{best_model_name}_history.png'):
    print(f"\n{best_model_name} Training History:")
    display(Image(f'{best_model_name}_history.png'))

## Step 17: Generate LaTeX Table for Report

In [ ]:
# Generate LaTeX table
print("\nLaTeX Table for Report:")
print("="*80)
print("\\begin{table}[h]")
print("\\centering")
print("\\begin{tabular}{lcccc}")
print("\\hline")
print("Model & RMSE & WRMSSE & Training Time (min) & Improvement \\\\")
print("\\hline")

for name in sorted(results.keys(), key=lambda x: results[x]['rmse']):
    r = results[name]
    t = training_times.get(name, 0) / 60
    imp = (baseline_rmse - r['rmse']) / baseline_rmse * 100
    imp_str = f"{imp:+.2f}\\%" if name != 'GRU' else "baseline"
    print(f"{name} & {r['rmse']:.4f} & {r['wrmsse']:.4f} & {t:.1f} & {imp_str} \\\\")

print("\\hline")
print("\\end{tabular}")
print("\\caption{Model Performance Comparison on M5 Dataset}")
print("\\label{tab:results}")
print("\\end{table}")
print("="*80)

---

## 🎉 Project Complete!

**All files saved to:**
- Local: `/content/`
- Google Drive: `/content/drive/MyDrive/m5_results/`

**For your report, you now have:**
- ✅ `final_results.json` - All metrics
- ✅ `model_comparison.png` - Visual comparison
- ✅ Training history plots for each model
- ✅ LaTeX table (generated above)
- ✅ Best model weights saved

**Next steps:**
1. Download all files from Google Drive
2. Write your final report using the results
3. Include visualizations and LaTeX table
4. Discuss improvements over baseline

**Good luck with your final submission! 🚀**